# Scratch Work for Flow Matching model 

In [9]:
from ase.io import read
from ase import Atoms
from ase.visualize import view

import numpy as np

import torch 
from torch import optim
from torch import nn
from torch_geometric.data import Data
from torch import Tensor

from collections import deque

from adsflow.models.gemnet.gemnet_flow import GemNetFlow
from adsflow.models.gemnet.gemnet_base import GemNetOC
from ase.io import read


## Compile GemNet for flow matching

In [ ]:
# GemNet-OC params
model_cfg = {
    "num_targets": 1,
    "num_spherical": 4,
    "num_radial": 64,
    "num_blocks": 2,
    "emb_size_atom": 128,
    "emb_size_edge": 256,
    "emb_size_trip_in": 64,
    "emb_size_trip_out": 64,
    "emb_size_quad_in": 32,
    "emb_size_quad_out": 32,
    "emb_size_aint_in": 64,
    "emb_size_aint_out": 64,
    "emb_size_rbf": 16,
    "emb_size_cbf": 16,
    "emb_size_sbf": 32,
    "num_before_skip": 2,
    "num_after_skip": 2,
    "num_concat": 1,
    "num_atom": 3,
    "num_output_afteratom": 3,
    "num_atom_emb_layers": 0,
    "num_global_out_layers": 2,
    "flow_matching": True,
    "use_pbc": True,
    "scale_backprop_forces": False,
    "cutoff": 12.0,
    "cutoff_qint": 12.0,
    "cutoff_aeaint": 12.0,
    "cutoff_aint": 12.0,
    "max_neighbors": 30,
    "max_neighbors_qint": 8,
    "max_neighbors_aeaint": 20,
    "max_neighbors_aint": 1000,
    "enforce_max_neighbors_strictly": True,
    "rbf": {"name": "gaussian"},
    "rbf_spherical": None,
    "envelope": {"name": "polynomial", "exponent": 5},
    "cbf": {"name": "spherical_harmonics"},
    "sbf": {"name": "legendre_outer"},
    "extensive": True,
    "forces_coupled": True,
    "output_init": "HeOrthogonal",
    "activation": "silu",
    "scale_file": None,
    "quad_interaction": True,
    "atom_edge_interaction": True,
    "edge_atom_interaction": True,
    "atom_interaction": True,
    "scale_basis": True,
}


In [3]:
GemNet = GemNetOC(**model_cfg)

# Count total number of parameters
trainable_params = sum(p.numel() for p in GemNet.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable_params}")

Trainable parameters: 37815808


## Testing GemNet-OC for energy/force regression

In [4]:
optimizer = optim.Adam(GemNet.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

# Training loop
def train():
    '''
    Defines the training procedure for the neural network
    '''
    l = []
    for epoch in range(1000): 
  
        # Create batch index (all atoms belong to structure 0)
        x1 = positions 
        batch = torch.zeros(len(x1), dtype=torch.long)

        # Build data objects
        data = Data(pos=x1, atomic_numbers=atomic_numbers, batch=batch)
        data.cell = torch.tensor(contcar.get_cell().array, dtype=torch.float32).unsqueeze(0) 
        data.pbc = torch.tensor([[True, True, True]])
        data.natoms = torch.tensor([len(positions)], dtype=torch.long)
        data.tags = torch.zeros(len(positions), dtype=torch.long)

        # Compute u_theta 
        output = GemNet(data)

        # Compute loss 
        force = torch.tensor([[0, 0, 1], [0, 0, -1]],  dtype=torch.float32)
        loss = loss_fn(output, force)
        l.append(loss.item())
        l_avg = np.mean(np.array(l[-50:]))
        print(f"Loss at epoch {epoch}: {loss.item()}")

        if loss.item() < 0.0000001:
            break 

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()


    return output


# AdsFlow Training Loop

In [11]:
def train(
    model,
    positions,
    atomic_numbers,
    cell,
    optimizer,
    loss_fn,
    num_epochs: int = 100,
    print_every: int = 10,
    tol: float = 1e-2,
    device: torch.device = torch.device("cpu"),
):
    
    model.train()
    # Move static tensors once
    x1 = positions.to(device)
    atomic_numbers = atomic_numbers.to(device)
    cell = cell.to(device)
    batch = torch.zeros(len(x1), dtype=torch.long, device=device)

    # Keep a fixed‐size buffer for running avg
    recent_losses = deque(maxlen=print_every)

    for epoch in range(num_epochs):
        # build data object 
        data = Data(
            atomic_numbers=atomic_numbers,    
            batch=batch,
            cell = cell, 
            pbc = torch.tensor([[True, True, True]]), 
            natoms = torch.tensor([len(positions)], dtype=torch.long), 
            tags = torch.zeros(len(positions), dtype=torch.long)
        )
        # interpolate between noise and sample: optimal transport
        x0 = torch.randn_like(x1, device=device)
        t  = torch.rand(1, device=device)
        xt = (1 - t) * x0 + t * x1
        data.t = t
        data.pos = xt

        optimizer.zero_grad()
        u_theta = model(data)
        target  = x1 - x0
        loss    = loss_fn(u_theta, target)
        loss.backward()
        optimizer.step()

        recent_losses.append(loss.item())
        if epoch % print_every == 0:
            avg = sum(recent_losses) / len(recent_losses)
            print(f"[{epoch:6d}] loss = {loss.item()}, avg({print_every}) = {avg}")

        if loss.item() < tol:
            print(f"Converged at epoch {epoch} (loss {loss.item()})")
            break

    return 



In [6]:
class BaselineVel(nn.Module):
    def __init__(self):
        super().__init__()
        # Input dim = flattened x_t (2 atoms × 3 dims) + t (1 dim) = 7
        self.net = nn.Sequential(
            nn.Linear(4 * 3 + 1, 64),
            nn.ReLU(),
            nn.Linear(64, 4 * 3),   # output = 2 atoms × 3 dims
        )

    def forward(self, data):
        x_flat = data.pos.view(1, -1)  # [1,6]
        t_flat = data.t.view(1, 1)     # [1,1]
        out = self.net(torch.cat([x_flat, t_flat], dim=1))
        return out.view(-1, 3)         # [2,3]

In [14]:
contcar = read("POSCAR.vasp")

x1 = torch.tensor(contcar.get_positions(), dtype=torch.float32)
z = torch.tensor(contcar.get_atomic_numbers(), dtype=torch.long)
cell = torch.tensor(contcar.get_cell().array, dtype=torch.float32).unsqueeze(0)
model = GemNetFlow(**model_cfg) # BaselineVel() #

train(model = model, 
      positions = x1,
      atomic_numbers = z, 
      cell = cell, 
      optimizer = optim.Adam(model.parameters(), 1e-3), 
      loss_fn = nn.MSELoss(), 
      tol = 1e-2, 
      print_every = 10, 
      num_epochs = 10000,
      )

[     0] loss = 0.545166552066803, avg(10) = 0.545166552066803
[    10] loss = 0.5590187907218933, avg(10) = 3.341664543747902
[    20] loss = 1.3739538192749023, avg(10) = 1.687092772871256
[    30] loss = 3.7157459259033203, avg(10) = 1.3344438552856446
[    40] loss = 1.174626111984253, avg(10) = 1.038015741109848
[    50] loss = 1.625139832496643, avg(10) = 0.8899345159530639
[    60] loss = 1.5879048109054565, avg(10) = 0.8344298020005226
[    70] loss = 0.4884144365787506, avg(10) = 0.5887399017810822
[    80] loss = 0.4743642807006836, avg(10) = 0.7136652797460556
[    90] loss = 0.26120826601982117, avg(10) = 1.342927598953247
[   100] loss = 0.7787286639213562, avg(10) = 1.1554024040699005
[   110] loss = 0.7369687557220459, avg(10) = 1.269029888510704
[   120] loss = 0.5788964629173279, avg(10) = 1.078837788105011
[   130] loss = 0.6029845476150513, avg(10) = 0.6093822486698628
[   140] loss = 0.4898664653301239, avg(10) = 0.5913537561893463
[   150] loss = 2.2365903854370117

KeyboardInterrupt: 

## Sampler

In [ ]:
# Sampler function

def step(model, xt: Tensor, t_start: Tensor, t_end: Tensor) -> Tensor: 

    # Define data object 
    data = Data() 
    data.atomic_numbers = z
    data.batch = torch.zeros(len(x1), dtype=torch.long)
    data.cell = torch.tensor(contcar.get_cell().array, dtype=torch.float32).unsqueeze(0) 
    data.pbc = torch.tensor([[True, True, True]])
    data.natoms = torch.tensor([len(xt)], dtype=torch.long)
    data.tags = torch.zeros(len(xt), dtype=torch.long)


    #t_start = t_start.view(1, 1).expand(xt.shape[0], 1)

    midpoint_time = t_start + (t_end - t_start) / 2 # evalute the midpoint 

    data.t = t_start
    data.pos = xt
    intermediate_value = model(data) # compute the velocity at this point using the trained neural network
    updated_xt = xt + intermediate_value * (t_end - t_start) / 2 # move xt using computed velocity 

    data.t = midpoint_time
    data.pos = updated_xt
    midpoint_output = model(data) # compute vt again 
    
    return xt + (t_end - t_start) * midpoint_output # compute xt

sample = False
if sample: 

    model.eval()  # Disables dropout/batch norm and ensures no gradients
    x0 = torch.randn_like(x1) 
    x = step(model, xt=x0, t_start=torch.Tensor([0]), t_end=torch.Tensor([1]))

    print(x0)
    print(x)
    
    # Sample from p1
    gen_struc = Atoms("NO3", positions=x.detach().numpy())

    view(gen_struc) 